# EDA de tiempos de cierre · NYC 311 *Noise - Residential* (2025)

**Proyecto:** NYC 311 Operational Analytics · V1
**Autor:** Junior E. Jiménez
**Datos:** `data/raw/2025-01.parquet` a `2025-12.parquet`, generados con `dataset.descargar_rango("2025-01", "2025-12")`

## Objetivo
Describir el tiempo de cierre de las solicitudes de ruido residencial de 2025, identificar diferencias por distrito, tipo de ruido y hora, y fijar las definiciones que usará V1: la partición temporal y el umbral de cierre tardío.

## Definiciones fijas del proyecto
Estas definiciones se establecen en este notebook y no cambian en el resto del proyecto.

| Concepto | Definición |
| --- | --- |
| Tiempo de cierre (`resolution_time_hours`) | `closed_date − created_date`, en horas |
| Partición | Entrenamiento: enero–agosto de 2025 · Validación: septiembre–octubre · Prueba: noviembre–diciembre (se usa una sola vez, en la semana 12) |
| Cierre tardío (`is_late`) | Tiempo de cierre mayor que el P75 de enero–agosto de 2025, redondeado a la media hora. Se calcula una vez y no se recalcula. |
| Supuesto de costo | Un cierre tardío no detectado (falso negativo) cuesta 3 veces más que una alerta innecesaria (falso positivo). Es un supuesto, no un dato; se usa en la semana 12. |

## Alcance y consideraciones
- El análisis exploratorio cubre el año completo; el umbral se calcula **solo** con enero–agosto.
- Las fechas no traen zona horaria (hora local de Nueva York). El cambio de horario puede desplazar una duración en una hora; no se corrige y se deja anotado.
- Herramientas: pandas para la preparación y los gráficos; DuckDB para las consultas de percentiles y ranking.
- Nombres de columnas en inglés y en *snake_case*, igual que las columnas de origen. Las variables derivadas de `created_date` llevan el prefijo `created_`.

## Configuración

In [84]:
%load_ext autoreload
%autoreload 2

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from nyc311_operational_analytics.config import FIGURES_DIR, RAW_DIR

conn = duckdb.connect()

sns.set_style("whitegrid")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Carga de datos
- Cargar los 12 archivos mensuales en una sola tabla.
- Verificar la cantidad de filas por mes contra el resumen de la descarga (total esperado: 463.349).
- Verificar los tipos de las columnas.
- Derivar las variables de trabajo: `resolution_time_hours`, `created_month`, `created_weekday` (0 = lunes … 6 = domingo) y `created_hour`.

In [85]:
df = pd.read_parquet(RAW_DIR)
print(f"Total de filas descargadas: {len(df)} filas")

Total de filas descargadas: 463349 filas


In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 463349 entries, 0 to 463348
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   unique_key              463349 non-null  object        
 1   created_date            463349 non-null  datetime64[ns]
 2   closed_date             463349 non-null  datetime64[ns]
 3   descriptor              463349 non-null  object        
 4   incident_zip            463333 non-null  object        
 5   borough                 463349 non-null  object        
 6   police_precinct         463349 non-null  object        
 7   latitude                462660 non-null  float64       
 8   longitude               462660 non-null  float64       
 9   open_data_channel_type  463349 non-null  object        
 10  status                  463349 non-null  object        
dtypes: datetime64[ns](2), float64(2), object(7)
memory usage: 38.9+ MB


In [87]:
df["resolution_time_hours"] = (
    df["closed_date"] - df["created_date"]
).dt.total_seconds() / 3600
df["created_month"] = df["created_date"].dt.month
df["created_weekday"] = df["created_date"].dt.weekday
df["created_hour"] = df["created_date"].dt.hour

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 463349 entries, 0 to 463348
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   unique_key              463349 non-null  object        
 1   created_date            463349 non-null  datetime64[ns]
 2   closed_date             463349 non-null  datetime64[ns]
 3   descriptor              463349 non-null  object        
 4   incident_zip            463333 non-null  object        
 5   borough                 463349 non-null  object        
 6   police_precinct         463349 non-null  object        
 7   latitude                462660 non-null  float64       
 8   longitude               462660 non-null  float64       
 9   open_data_channel_type  463349 non-null  object        
 10  status                  463349 non-null  object        
 11  resolution_time_hours   463349 non-null  float64       
 12  created_month           463349

In [89]:
conn.sql("""
    SELECT
         created_month,
         COUNT(*) AS total_filas
    FROM df
    GROUP BY created_month
    ORDER BY created_month ASC
""").df()

,created_month,total_filas
0,1,70837
1,2,26945
2,3,28850
3,4,28019
4,5,32905
5,6,29740
6,7,26659
7,8,32756
8,9,36142
9,10,49182


**Hallazgos:**

- Se cargaron 463.349 solicitudes de 2025. La cantidad por mes coincide exactamente con el resumen de la descarga, así que no se perdieron filas al leer los archivos.
- Tipos correctos: `created_date` y `closed_date` como `datetime64`, `latitude` y `longitude` como `float64`, y los identificadores (`unique_key`, `incident_zip`, `police_precinct`) como texto.
- El volumen varía mucho entre meses: febrero–agosto se mantiene entre 26.659 y 32.905, el último trimestre sube hasta 57.386 en diciembre, y enero (70.837) más que duplica a febrero. Enero se investiga en la sección 3.4 antes de calcular el umbral.


## 2. Limpieza de datos
Aplicar las reglas de limpieza obligatorias y registrar cada decisión con el número de filas afectadas. Las reglas se aplican en el orden de la tabla.

| # | Regla | Filas afectadas | Decisión | Justificación |
| --- | --- | --- | --- | --- |
| 2.1 | Solicitudes sin `closed_date` | 0 | No aplica | No hay solicitudes sin `closed_date` en 2025. |
| 2.2 | Cierre anterior o igual a la creación | 39 | Eliminar | Las 39 se crearon el 2025-11-02 entre la 1:00 y las 2:00, la hora que se repite al terminar el horario de verano. Sin zona horaria no se puede conocer la duración real. Se eliminan (0.008 %) y el efecto se documenta como limitación. |
| 2.3 | Tiempo de cierre mayor de 7 días (revisar antes de decidir) | 150 | Eliminar | 150 solicitudes (0.03 %) de 7 a 65 días, creadas casi todas en marzo de 2025. El 90 % se cerró en 4 fechas (66 el 2025-04-09, 29 el 04-11, 22 el 04-10 y 18 el 03-19): cierres administrativos en lote que no reflejan el tiempo real de atención. Se eliminan. |
| 2.4 | Duplicados por `unique_key` | 0 | No aplica | No hay solicitudes con `unique_key` duplicados. |
| 2.5 | Distrito `Unspecified` | 23 | Eliminar | 23 filas (0.005 %) sin distrito. El distrito es una variable de análisis y del modelo; sin él, la fila no se puede usar. Regla obligatoria del proyecto. |
| 2.6 | `status` distinto de `Closed` | 139 | Mantener | 139 filas con `status` "Unspecified" (137 del Bronx). Tienen `closed_date`, así que el tiempo de cierre se puede calcular; el `status` por sí solo no invalida la fila. Ojo: su mediana es de 9 segundos y 120 de ellas cierran en menos de 1 minuto, así que salen en la regla 2.7. |
| 2.7 | Cierres en menos de 1 minuto | 3.921 | Eliminar | 3.921 filas (0.85 %), mediana de 36 segundos. Se concentran en un solo lugar: el 76 % es del Precinto 47 (código postal 10466, Bronx) y el 80 % entró por app móvil, contra el 42 % del total. Es el patrón de cierres automáticos o duplicados, no de una atención real. |

**Resultado de la limpieza:** 463.349 filas iniciales · 459.216 filas finales · 4.133 excluidas (0.89 %).

### 2.1 Sin `closed_date`

In [90]:
raw_df = df.copy()
print(f"Filas iniciales: {len(raw_df):,}")

Filas iniciales: 463,349


In [91]:
df["closed_date"].isnull().sum()

np.int64(0)

### 2.2 Cierre anterior o igual a la creación

In [92]:
cierres_anteriores = df[df["closed_date"] <= df["created_date"]]
print(f"Cierres anteriores o iguales a la creación: {len(cierres_anteriores):,}")

Cierres anteriores o iguales a la creación: 39


In [93]:
cambio_horario = cierres_anteriores[
    (cierres_anteriores["created_date"] >= "2025-11-02 01:00:00")
    & (cierres_anteriores["created_date"] < "2025-11-02 02:00:00")
]
print(f"De ellos, en la hora repetida del cambio de horario: {len(cambio_horario):,}")

De ellos, en la hora repetida del cambio de horario: 39


In [94]:
n_antes = len(df)
df = df[df["closed_date"] > df["created_date"]]
print(f"Regla 2.2: se eliminan {n_antes - len(df):,} · quedan {len(df):,}")

Regla 2.2: se eliminan 39 · quedan 463,310


### 2.3 Tiempo de cierre mayor de 7 días

In [95]:
mayor7 = df[df["resolution_time_hours"] > 168]  # 168 h = 7 días
mayor7["resolution_time_hours"].describe()

count     150.000000
mean      516.959854
std       348.034463
min       172.385278
25%       250.740139
50%       413.700972
75%       663.478611
max      1566.813889
Name: resolution_time_hours, dtype: float64

In [96]:
# Validar los que cerraron en el mismo dia
mayor7["closed_date"].dt.date.value_counts()

closed_date
2025-04-09    66
2025-04-11    29
2025-04-10    22
2025-03-19    18
2025-03-09     5
2025-03-06     2
2025-03-18     2
2025-03-23     2
2025-08-03     2
2025-03-02     1
2025-07-18     1
Name: count, dtype: int64

In [97]:
n_antes = len(df)
df = df[df["resolution_time_hours"] <= 168]
print(f"Regla 2.3: se eliminan {n_antes - len(df):,} · quedan {len(df):,}")

Regla 2.3: se eliminan 150 · quedan 463,160


### 2.4 Duplicados por `unique_key`

In [98]:
duplicados = df["unique_key"].duplicated().sum()
print(f"Filas con unique_key duplicado: {duplicados:,}")

Filas con unique_key duplicado: 0


### 2.5 Distrito `Unspecified`

In [99]:
unspecified = df[df["borough"] == "Unspecified"]
print(f"Filas con distrito Unspecified: {len(unspecified):,}")

Filas con distrito Unspecified: 23


In [100]:
n_antes = len(df)
df = df[df["borough"] != "Unspecified"]
print(f"Regla 2.5: se eliminan {n_antes - len(df):,} · quedan {len(df):,}")

Regla 2.5: se eliminan 23 · quedan 463,137


### 2.6 `status` distinto de `Closed`

In [101]:
df["status"].value_counts()

status
Closed         462998
Unspecified       139
Name: count, dtype: int64

In [102]:
s_uspecified = df[df["status"] == "Unspecified"]
s_uspecified["borough"].value_counts()

borough
BRONX        137
BROOKLYN       1
MANHATTAN      1
Name: count, dtype: int64

In [103]:
s_uspecified["resolution_time_hours"].describe()

count    139.000000
mean       0.144858
std        1.027109
min        0.000833
25%        0.001667
50%        0.002500
75%        0.008056
max        8.643611
Name: resolution_time_hours, dtype: float64

### 2.7 Cierres en menos de 1 minuto

In [ ]:
menos_1min = df[df["resolution_time_hours"] < 1 / 60]
print(
    f"Cierres en menos de 1 minuto: {len(menos_1min):,} ({len(menos_1min) / len(df):.2%})"
)
(menos_1min["resolution_time_hours"] * 3600).describe().round(1)  # en segundos

Cierres en menos de 1 minuto: 3,921 (0.85%)


count    3921.0
mean       36.2
std        13.7
min         3.0
25%        26.0
50%        36.0
75%        47.0
max        59.0
Name: resolution_time_hours, dtype: float64

In [105]:
# ¿Se concentran en algún distrito o canal? Proporción en el total contra proporción en el grupo
for col in ["borough", "open_data_channel_type"]:
    comparacion = pd.concat(
        {
            "total": df[col].value_counts(normalize=True),
            "menos_1min": menos_1min[col].value_counts(normalize=True),
        },
        axis=1,
    ).round(3)
    display(comparacion)

,total,menos_1min
borough,,
BRONX,0.452,0.810
BROOKLYN,0.224,0.080
QUEENS,0.161,0.060
MANHATTAN,0.141,0.045
STATEN ISLAND,0.022,0.005


,total,menos_1min
open_data_channel_type,,
ONLINE,0.457,0.170
MOBILE,0.420,0.803
PHONE,0.123,0.027
UNKNOWN,0.000,0.001
OTHER,0.000,NaN


In [106]:
menos_1min["police_precinct"].value_counts().head(5)

police_precinct
Precinct 47     2995
Precinct 107     113
Precinct 1        86
Precinct 84       79
Precinct 42       75
Name: count, dtype: int64

In [107]:
# Cruce con la regla 2.6
menos_1min["status"].value_counts()

status
Closed         3801
Unspecified     120
Name: count, dtype: int64

In [108]:
n_antes = len(df)
df = df[df["resolution_time_hours"] >= 1 / 60]
print(f"Regla 2.7: se eliminan {n_antes - len(df):,} · quedan {len(df):,}")

Regla 2.7: se eliminan 3,921 · quedan 459,216


### Resultado de la limpieza

In [109]:
filas_iniciales = len(raw_df)
filas_finales = len(df)
excluidas = filas_iniciales - filas_finales
print(f"Filas iniciales: {filas_iniciales:,}")
print(f"Filas finales: {filas_finales:,}")
print(f"Excluidas: {excluidas:,} ({excluidas / filas_iniciales:.2%})")

Filas iniciales: 463,349
Filas finales: 459,216
Excluidas: 4,133 (0.89%)


**Hallazgos:**

- Se excluyeron 4.133 filas (0.89 %); quedan 459.216 para el análisis.
- Ninguna exclusión responde a errores de captura sencillos: 39 filas son efecto del cambio de horario (2025-11-02), 150 son cierres administrativos en lote de un atraso de marzo, 23 no tienen distrito y 3.921 se cerraron en menos de 1 minuto.
- Los cierres en menos de 1 minuto se concentran en el Precinto 47 (código postal 10466, Bronx): el 76 % de ellos, y el 80 % entró por app móvil, contra el 42 % del total.
- Las 139 filas con `status` "Unspecified" se mantuvieron por esa regla, pero 120 de ellas salieron por la 2.7: el `status` no era el problema, sino el cierre instantáneo.
- Limitación: las duraciones que cruzan los cambios de horario (9 de marzo y 2 de noviembre de 2025) pueden tener una hora de más o de menos. Sin zona horaria no se pueden corregir; el efecto sobre los percentiles es despreciable.

## 3. Volumen de solicitudes
Describir cómo se distribuye el volumen en el tiempo y entre distritos, e investigar la concentración de solicitudes del Bronx en enero antes de calcular el umbral.

### 3.1 Solicitudes por mes

### 3.2 Solicitudes por distrito

### 3.3 Solicitudes por mes y distrito

### 3.4 Investigación: Bronx en enero
Determinar si el volumen del Bronx en enero se concentra en pocos días, en pocas ubicaciones o en un patrón de cierre distinto, y decidir cómo se trata en el análisis y en el cálculo del umbral.

**Decisión sobre el Bronx en enero:**

**Hallazgos:**

- 
- 

## 4. Distribución del tiempo de cierre
Caracterizar la forma de la distribución y sus colas con los datos limpios.

### 4.1 Medidas de localización y dispersión
Media, mediana, desviación estándar, IQR, P90, P95 y P99.

### 4.2 Forma de la distribución

### 4.3 Colas
Proporción de solicitudes que cierran en más de 12, 24 y 72 horas.

**Hallazgos:**

- 
- 

## 5. Diferencias por segmento
Para cada segmento: cantidad de solicitudes, mediana, P75 y P90 del tiempo de cierre, y un gráfico comparativo.

### 5.1 Por distrito

### 5.2 Por tipo de ruido (`descriptor`)

### 5.3 Por hora del día y día de la semana

### 5.4 Ranking de precintos
Los cinco precintos con mayor mediana de tiempo de cierre dentro de cada distrito, considerando solo precintos con un volumen mínimo definido y justificado en el análisis.

**Hallazgos:**

- 
- 

## 6. Partición y umbral de cierre tardío

### 6.1 Partición temporal
Asignar cada solicitud a entrenamiento, validación o prueba y reportar el tamaño de cada partición.

### 6.2 Umbral de cierre tardío
P75 del tiempo de cierre en entrenamiento (enero–agosto), redondeado a la media hora.

### 6.3 Proporción de cierres tardíos por partición

### 6.4 Sensibilidad del umbral
Comparar el P75 de entrenamiento con y sin el bloque identificado en la sección 3.4, y justificar el valor adoptado.

**Definiciones adoptadas para V1** (se copian al README):

| Concepto | Valor |
| --- | --- |
| Filas de entrenamiento / validación / prueba | |
| Umbral de cierre tardío | h |
| % de cierres tardíos en entrenamiento | |

## 7. Conclusiones

### Resumen
- 
- 
- 

### Decisiones para V1
- 
- 

### Limitaciones
- 
- 

### Próximos pasos
- 
- 